In [3]:
import numpy as np
import pandas as pd
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.ndimage import gaussian_filter



In [4]:
def stochastic_seir(N, R0, sigma, gamma, I0=1, E0=0, max_time=365):

    beta = R0 * gamma

    S = N - I0 - E0
    E = E0
    I = I0
    R = 0

    t = 0
    times = [t]
    S_series = [S]

    while t < max_time and I > 0:

        infection = beta * S * I / N
        incubation = sigma * E
        recovery = gamma * I

        rate_total = infection + incubation + recovery

        if rate_total == 0:
            break

        dt = np.random.exponential(1 / rate_total)
        t += dt

        r = np.random.rand() * rate_total

        if r < infection:
            S -= 1
            E += 1

        elif r < infection + incubation:
            E -= 1
            I += 1

        else:
            I -= 1
            R += 1

        times.append(t)
        S_series.append(S)

    return np.array(times), np.array(S_series)

In [5]:
def outbreak_duration(times, S, N, R0):

    Rt = R0 * (S / N)

    active = Rt > 1

    if np.sum(active) == 0:
        return 0

    start = times[np.where(active)[0][0]]
    end = times[np.where(active)[0][-1]]

    return end - start

In [12]:
def build_phase_diagram():

    R0_grid = np.linspace(1.05, 3.0, 40)
    N_grid = np.logspace(3, 7, 40).astype(int)

    sigma = 1/1.5
    gamma = 1/3

    runs_per_cell = 30

    surface = np.zeros((len(N_grid), len(R0_grid)))

    for i, N in enumerate(N_grid):

        for j, R0 in enumerate(R0_grid):

            durations = []

            for k in range(runs_per_cell):

                times, S = stochastic_seir(
                    N=N,
                    R0=R0,
                    sigma=sigma,
                    gamma=gamma
                )

                d = outbreak_duration(times, S, N, R0)
                durations.append(d)

            surface[i, j] = np.mean(durations)

    return R0_grid, N_grid, surface

In [14]:

R0_grid, N_grid, surface = build_phase_diagram()
surface = gaussian_filter(surface, sigma=1)


plt.figure(figsize=(10,7))

plt.imshow(
    surface,
    origin="lower",
    aspect="auto",
    extent=[
        R0_grid.min(),
        R0_grid.max(),
        np.log10(N_grid.min()),
        np.log10(N_grid.max())
    ]
)

plt.colorbar(label="Mean outbreak duration (days)")

plt.xlabel("R0")
plt.ylabel("log10 population")

plt.title("Smooth phase diagram of outbreak duration")

plt.show()

KeyboardInterrupt: 